# E-Commerce Analytics End-to-End Case Study (Python)

In [193]:
custs = pd.read_csv("CUSTOMERS.csv")
sells = pd.read_csv("SELLERS.csv")
geolocs = pd.read_csv("GEO_LOCATION.csv")
prods = pd.read_csv("PRODUCTS.csv")
ords = pd.read_csv("ORDERS.csv")
ord_items = pd.read_csv("ORDER_ITEMs.csv")
ord_rev_rats = pd.read_csv("ORDER_REVIEW_RATINGS.csv")
ord_pays = pd.read_csv("ORDER_PAYMENTS.csv")


In [194]:
custs.shape

(99441, 5)

In [195]:
ords.shape

(99441, 8)

In [196]:
df = pd.merge(left = custs, right = ords, left_on = "customer_id", right_on = "customer_id", how = 'left')

In [197]:
df.shape

(99441, 12)

In [198]:
geolocs.columns

Index(['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng',
       'geolocation_city', 'geolocation_state'],
      dtype='object')

In [199]:
sells.columns

Index(['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state'], dtype='object')

In [200]:
df = custs.merge(ords, on = "customer_id", how = 'left')\
      .merge(ord_items, on = "order_id", how = 'left')\
      .merge(ord_rev_rats, on = "order_id", how = 'left')\
      .merge(ord_pays, on = "order_id", how = 'left')\
      .merge(prods, on = "product_id", how = 'left')\
      .merge(sells, on = "seller_id", how = 'left')\
      .merge(geolocs, left_on = "seller_zip_code_prefix", right_on = "geolocation_zip_code_prefix", how = 'left')

In [201]:
df.shape

(119151, 42)

In [202]:
df.sample(3)

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,...,product_height_cm,product_width_cm,seller_zip_code_prefix,seller_city,seller_state,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
4050,4c2352fefc6c76c844fb918aa3a1478f,120abf6e9c05f5ba0d69603dc05a98cd,38600,Bhatapara,Chhattisgarh,6c6e590c8ce02ecc30bef8bd3b34d82f,delivered,1/29/2017 20:37,1/29/2017 21:41,1/30/2017 8:27,...,15.0,52.0,89204.0,Pithora,Chhattisgarh,89204.0,-26.283149,-48.851285,Pithora,Chhattisgarh
116283,0b524da779a488d1550b3e717bfe03d5,eea7c506bed0997817f14da59e862db1,75555,Maholi,Uttar Pradesh,6639717557f610a34aec7003122f7c33,delivered,5/29/2017 21:18,5/31/2017 6:45,6/13/2017 11:09,...,12.0,37.0,16301.0,Vitthal Udyognagar INA,Gujarat,16301.0,-21.426570,-50.067918,Vitthal Udyognagar INA,Gujarat
18189,c3af7c0e51b2768d53a7654b57fc0492,692e54a1377629febd7d56cd3bb6ace4,6608,Ambagarh Chowki,Chhattisgarh,34fa4dce8775b3b4accea9779a4a6562,delivered,3/19/2018 15:04,3/19/2018 15:24,3/20/2018 21:25,...,6.0,36.0,4782.0,Akkarampalle,Andhra Pradesh,4782.0,-23.693986,-46.701883,Akkarampalle,Andhra Pradesh


In [203]:
df.isnull().sum()

customer_id                         0
customer_unique_id                  0
customer_zip_code_prefix            0
customer_city                       0
customer_state                      0
order_id                            0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 177
order_delivered_carrier_date     2086
order_delivered_customer_date    3421
order_estimated_delivery_date       0
order_item_id                     833
product_id                        833
seller_id                         833
shipping_limit_date               833
price                             833
freight_value                     833
review_id                           0
review_score                        0
review_creation_date                0
review_answer_timestamp             0
payment_sequential                  3
payment_type                        3
payment_installments                3
payment_value                       3
product_cate

In [204]:
df.duplicated().sum()

np.int64(0)

In [205]:
df['order_purchase_timestamp'] = pd.to_datetime(df['order_purchase_timestamp'])

## 1. Perform Detailed exploratory analysis
* a. Define & calculate high level metrics like (Total Revenue, Total quantity, Total
  products, Total categories, Total sellers, Total locations, Total channels, Total
  payment methods etc…)
* b. Understanding how many new customers acquired every month
* c. Understand the retention of customers on month on month basis
* d. How the revenues from existing/new customers on month on month basis
* e. Understand the trends/seasonality of sales, quantity by category, location, month,
  week, day, time, channel, payment method etc…
* f. Popular Products by month, seller, state, category.
* g. Popular categories by state, month
* h. List top 10 most expensive products sorted by price

### a. Define & calculate high level metrics like (Total Revenue, Total quantity, Total products, Total categories, Total sellers, Total locations, Total channels, Total payment methods etc…)

In [206]:
# Total Revenue
total_revenue = df['payment_value'].sum()
total_revenue

np.float64(20581109.619999997)

In [207]:
# Total Quantity
total_quantity = df.order_id.count()
total_quantity

np.int64(119151)

In [208]:
# Total Products
total_products = df["product_id"].nunique()
total_products

32951

In [209]:
# Total Categories
df.product_category_name.nunique()

71

In [210]:
# Total Sellers
df.seller_id.nunique()

3095

In [211]:
# Total Locations
df.geolocation_city.nunique()

529

In [212]:
# Total Channels


In [213]:
# Total Payment Methods ...etc
df.payment_type.nunique()

5

In [214]:
# Total Orders
total_orders = df['order_id'].nunique()
total_orders

99441

In [215]:
# Total Customers
total_customers = df['customer_id'].nunique()
total_customers

99441

### b. Understanding how many new customers acquired every month

In [216]:
df["month"] = df["order_purchase_timestamp"].dt.month_name()
new_customers = df.groupby("month")["customer_id"].nunique()
new_customers
# nsight :Company is acquiring more customers every month = good marketing performance

month
April         9343
August       10843
December      5674
February      8508
January       8069
July         10318
June          9412
March         9893
May          10573
November      7544
October       4959
September     4305
Name: customer_id, dtype: int64

### c. Understand the retention of customers on month on month basis

In [217]:
# Retention = Customers who purchased more than once.

customer_orders = df.groupby("customer_id")["order_id"].nunique()
repeat_customers = customer_orders[customer_orders > 1].count()
retention_rate = repeat_customers / total_customers
retention_rate
# High retention means customers are satisfied but here low retention means customer are not satisfied.

np.float64(0.0)

### d. How the revenues from existing Vs new customers on month on month basis

In [218]:
df = df.sort_values(by=['customer_id','order_purchase_timestamp'])

df['order_number'] = df.groupby('customer_id').cumcount()+1

new_customer_revenue = df[df['order_number']==1]['payment_value'].sum()
print("new_customer_revenue :",new_customer_revenue)

existing_customer_revenue = df[df['order_number']>1]['payment_value'].sum()
print("existing_customer_revenue :",existing_customer_revenue)

new_customer_revenue : 15744810.659999998
existing_customer_revenue : 4836298.96


### e. Understand the trends/seasonality of sales, quantity by category, location, month, week, day, time, channel, payment method etc…

In [219]:
trend_seasonality = df.groupby(["product_category_name","geolocation_state","month"])\
                                     .agg(Sales = ("payment_value","sum"), Quantity = ("order_item_id","count"))
trend_seasonality
# Sales increase during festival months

Sales  Quantity
product_category_name      geolocation_state month                       
Agro_Industry_And_Commerce Andhra Pradesh    April      4596.10        10
                                             August     6196.42        12
                                             December   6261.36         9
                                             February  10834.19        56
                                             January    6805.12        19
...                                                         ...       ...
Watches_Gifts              Tamil Nadu        October     263.01         1
                           Uttar Pradesh     February    667.88         1
                                             January    1125.90         2
                                             March       600.75         1
                                             October    1999.03         3

[3819 rows x 2 columns]

In [220]:
monthly_sales = df.groupby('month')['payment_value'].sum()
monthly_sales

month
April        2012546.68
August       2133110.99
December     1061175.98
February     1679493.75
January      1616267.67
July         2111509.92
June         1913460.58
March        2034439.48
May          2249436.06
November     1610581.21
October      1123652.21
September    1035435.09
Name: payment_value, dtype: float64

### f. Popular Products by month, seller, state, category.

In [221]:
top_products = df.groupby(["product_category_name", "seller_id", "month","seller_state"])["order_id"].count().sort_values(ascending = False).head(10)
top_products

product_category_name  seller_id                         month     seller_state  
Garden_Tools           1f50f920176fa81dab994f9023523100  November  Andhra Pradesh    371
Watches_Gifts          6560211a19b47992c3666cc44a7e94c0  July      Andhra Pradesh    267
                                                         August    Andhra Pradesh    247
Bed_Bath_Table         4a3ca9315b744ce9f8e9374361493884  May       Gujarat           226
Office_Furniture       7c67e1448b00f6e969d365cea6b010ab  March     Andhra Pradesh    214
Watches_Gifts          6560211a19b47992c3666cc44a7e94c0  June      Andhra Pradesh    206
Stationery             3d871de0142ce09b7081e2b9d1733cb1  January   Andhra Pradesh    205
Watches_Gifts          7d13fca15225358621be4086e1eb0964  May       Andhra Pradesh    205
Garden_Tools           1f50f920176fa81dab994f9023523100  February  Andhra Pradesh    190
Telephony              ea8482cd71df3c1969d7b9473ff13abc  January   Andhra Pradesh    188
Name: order_id, dtype: int64

### g. Popular categories by state, month

In [222]:
df.groupby(["product_category_name","seller_state", "month",])["order_id"].count().sort_values(ascending = False).head(10)

product_category_name  seller_state    month   
Computers_Accessories  Andhra Pradesh  February    974
Health_Beauty          Andhra Pradesh  August      964
                                       June        962
                                       May         886
                                       July        880
                                       April       759
Computers_Accessories  Andhra Pradesh  March       741
Bed_Bath_Table         Gujarat         May         726
                                       June        722
                                       July        715
Name: order_id, dtype: int64

### h. List top 10 most expensive products sorted by price

In [223]:
expensive_products = df[['product_id','price']].drop_duplicates()
expensive_products = expensive_products.sort_values(by='price', ascending=False).head(10)
expensive_products

,product_id,price
16177,489ae2aa008f021502940f251d4cce7f,6735.00
25418,69c590f7ffc7bf8db97190b6cb6ed62e,6729.00
4700,1bdf5e6731585cf01aa8169c7028d6ad,6499.00
32766,a6492cc69376c469ab6f61d8f44de961,4799.00
14462,c3ed642d592594bb648ff4a04cee2747,4690.00
48803,259037a6a41845e455183f89c5035f18,4590.00
77294,a1beef8f3992dbd4cd8726796aa69c53,4399.87
7094,6cdf8fc1d741c76586d8b6b15e9eef30,4099.99
108144,dd113cb02b2af9c8e5787e8f1f0722f6,4059.00
100520,6902c1962dd19d540807d0ab8fade5c6,3999.90


## 2. Performing Customers/sellers Segmentation
* a. Divide the customers into groups based on the revenue generated
* b. Divide the sellers into groups based on the revenue generated 

In [224]:
# a. Divide the customers into groups based on the revenue generated
customer_revenue = df.groupby('customer_id')['payment_value'].sum()
customer_revenue = customer_revenue.sort_values()
customer_revenue

customer_id
197a2a6a77da93f678ea0d379f21da0a         0.00
86dc2ffce2dfff336de2f386a786e574         0.00
3532ba38a3fd242259a514ac2b6ae6b6         0.00
a73c1f73f5772cf801434bf984b0b1a7         0.00
a790343ca6f3fee08112d678b43aa7c5         9.59
                                      ...    
1ff773612ab8934db89fd5afa8afe506     30186.00
05455dfa7cd02f13d132aa7a6a9729c6     36489.24
be1b70680b9f9694d8c70f41fa3dc92b     44048.00
bd5d39761aa56689a265d95d8d32b8be     45256.00
1617b1357756262bfa56ab541c47bc16    109312.64
Name: payment_value, Length: 99441, dtype: float64

In [225]:
# b. Divide the sellers into groups based on the revenue generated
seller_revenue = df.groupby('seller_id')['payment_value'].sum()
seller_revenue = seller_revenue.sort_values(ascending=False)
seller_revenue

seller_id
7c67e1448b00f6e969d365cea6b010ab    512645.19
1025f0e2d44d7041d6cf58b6550e0bfa    312456.49
4a3ca9315b744ce9f8e9374361493884    306138.80
1f50f920176fa81dab994f9023523100    291918.98
53243585a1d6dc2643021fd1853d8905    284903.08
                                      ...    
ad14615bdd492b01b0d97922e87cb87f        19.21
702835e4b785b67a084280efca355756        18.56
4965a7002cca77301c82d3f91b82e1a9        16.36
77128dec4bec4878c37ab7d6169d6f26        15.22
cf6f6bc4df3999b9c6440f124fb2f687        12.22
Name: payment_value, Length: 3095, dtype: float64

## 3. Cross-Selling (Which products are selling together)
* Hint: We need to find which of the top 10 combinations of products are selling together in each transaction. (combination of 2 or 3 buying together)

In [226]:
from itertools import combinations

order_product = df.groupby('order_id')['product_id'].apply(list)

combo_counts = {}

for items in order_product:
    
    for combo in combinations(items,2):
        
        combo_counts[combo] = combo_counts.get(combo,0)+1

cross_selling = pd.DataFrame(combo_counts.items(), columns=['product_pair','count'])

cross_selling.sort_values(by='count', ascending=False).head(10)
# Insight:
# Products often bought together → bundle offer opportunity

,product_pair,count
6392,"(ebf9bc6cd600eadd681384e3116fda85, 5ddab10d5e0...",882
4593,"(ebf9bc6cd600eadd681384e3116fda85, ebf9bc6cd60...",862
10980,"(0554911df28fda9fd668ce5ba5949695, 0554911df28...",703
10832,"(1aecdb5fa3add74e385f25c6c527a462, 1aecdb5fa3a...",406
61,"(422879e10f46682990de24d770e7f83d, 422879e10f4...",326
9133,"(8d37ee446981d3790967d0268d6cfc81, 8d37ee44698...",325
7397,"(4e53a453045707bbc5febcf5f32097ac, 4e53a453045...",276
5240,"(eea3e07f864a0a1389726d8a5f31c3f6, eea3e07f864...",276
3558,"(11250b0d4b709fee92441c5f34122aed, 11250b0d4b7...",276
8837,"(0449db5eede617c5fd413071d582f038, 0449db5eede...",276


## 4. Payment Behaviour
* a. How customers are paying?
* b. Which payment channels are used by most customers?


In [227]:
# a. How customers are paying?
payment_method = df['payment_type'].value_counts()
payment_method
# Insight:
# Most used payment method = credit_card    

payment_type
credit_card    87784
UPI            23190
voucher         6465
debit_card      1706
not_defined        3
Name: count, dtype: int64

In [228]:
# b. Which payment channels are used by most customers?
payment_method.head(1)

payment_type
credit_card    87784
Name: count, dtype: int64

## 5. Customer satisfaction towards category & product
* a. Which categories (top 10) are maximum rated & minimum rated?
* b. Which products (top10) are maximum rated & minimum rated?
* c. Average rating by location, seller, product, category, month etc.
 Etc..

In [229]:
# a. Which categories (top 10) are maximum rated & minimum rated?

max_rated_categories = df.groupby('product_category_name')['review_score'].mean()
max_rated_categories = max_rated_categories.sort_values(ascending=False).head(10)
print("MAXIMUM_Rated_Categories : \n\n",max_rated_categories)

min_rated_categories = df.groupby('product_category_name')['review_score'].mean()
min_rated_categories = min_rated_categories.sort_values(ascending=False).tail(10)
print("MINIMUM_Rated_Categories : \n\n",max_rated_categories)

MAXIMUM_Rated_Categories : 

 product_category_name
Cds_Dvds_Musicals                        4.642857
Fashion_Childrens_Clothes                4.500000
Books_General_Interest                   4.431858
Books_Imported                           4.419355
Books_Technical                          4.345588
Costruction_Tools_Tools                  4.333333
Small_Appliances_Home_Oven_And_Coffee    4.320513
Food_Drink                               4.312715
Luggage_Accessories                      4.290628
Fashion_Sport                            4.258065
Name: review_score, dtype: float64
MINIMUM_Rated_Categories : 

 product_category_name
Cds_Dvds_Musicals                        4.642857
Fashion_Childrens_Clothes                4.500000
Books_General_Interest                   4.431858
Books_Imported                           4.419355
Books_Technical                          4.345588
Costruction_Tools_Tools                  4.333333
Small_Appliances_Home_Oven_And_Coffee    4.320513
Food_Drink 

In [230]:
# b. Which products (top10) are maximum rated & minimum rated?

max_rated_products = df.groupby('product_id')['review_score'].mean()
max_rated_products = max_rated_categories.sort_values(ascending=False).head(10)
print("MAXIMUM_Rated_Products : \n\n",max_rated_products)

min_rated_products = df.groupby('product_id')['review_score'].mean()
min_rated_products = min_rated_products.sort_values(ascending=False).tail(10)
print("MINIMUM_Rated_Products : \n\n",max_rated_categories)

MAXIMUM_Rated_Products : 

 product_category_name
Cds_Dvds_Musicals                        4.642857
Fashion_Childrens_Clothes                4.500000
Books_General_Interest                   4.431858
Books_Imported                           4.419355
Books_Technical                          4.345588
Costruction_Tools_Tools                  4.333333
Small_Appliances_Home_Oven_And_Coffee    4.320513
Food_Drink                               4.312715
Luggage_Accessories                      4.290628
Fashion_Sport                            4.258065
Name: review_score, dtype: float64
MINIMUM_Rated_Products : 

 product_category_name
Cds_Dvds_Musicals                        4.642857
Fashion_Childrens_Clothes                4.500000
Books_General_Interest                   4.431858
Books_Imported                           4.419355
Books_Technical                          4.345588
Costruction_Tools_Tools                  4.333333
Small_Appliances_Home_Oven_And_Coffee    4.320513
Food_Drink     

In [231]:
# c. Average rating by location, seller, product, category, month etc. Etc..
AVG_Rat = df.groupby(["seller_id","seller_state","product_category_name","product_id","month"])["review_score"].mean().sort_values(ascending = False)
AVG_Rat

seller_id                         seller_state    product_category_name  product_id                        month    
ffff564a4f9085cd26170f4732393726  Andhra Pradesh  Market_Place           de6517dda8e49774f58c07f80abc8d7a  October      5.0
                                                  Electronics            c4b925e40f11289063a854c47aaef129  January      5.0
0015a82c2db000af6aaaf3ae2ecb0532  Andhra Pradesh  Small_Appliances       a2ff5a97bf95719e38ea2e3b4105bce8  September    5.0
fffd5413c0700ac820c7069d66d98c89  Haryana         Housewares             bec2975b905c2d5a70fd97abf8c894e3  June         5.0
                                                                         ac850a749748b386d598124ae34ba1b0  January      5.0
                                                                                                                       ... 
4830e40640734fc1c52cd21127c341d4  Andhra Pradesh  Housewares             20d5583ac76a88720b002f353afec2d0  July         1.0
               

## Thanks